# Arcadia: can TRELLIS run on a Kaggle T4? (time-boxed)

TRELLIS (Microsoft, MIT licence) is a stronger image-to-3D model than TripoSR, but it was built for A100-class GPUs (it asks for 16 GB, a T4 has 15) and needs several CUDA extensions compiled. This notebook gives the install **6 minutes**, then tries the temple and the ship from the assets run (attached as an input). Whatever happens is written to `/kaggle/working/trellis/status.json`.

In [ ]:
import os, subprocess, sys, json, time, glob
os.makedirs("/kaggle/working/trellis", exist_ok=True)
status = {"gpu": subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()}
print(status)
def done(**kw):
    status.update(kw); json.dump(status, open("/kaggle/working/trellis/status.json", "w"), indent=1); print(status)

In [ ]:
t = time.time()
os.environ["ATTN_BACKEND"] = "xformers"   # flash-attn needs Ampere or newer; a T4 is Turing
os.environ["SPCONV_ALGO"] = "native"
steps = [
  "git clone -q --recurse-submodules --depth 1 https://github.com/microsoft/TRELLIS.git /kaggle/working/TRELLIS",
  "pip install -q easydict plyfile utils3d@git+https://github.com/EasternJournalist/utils3d.git rembg onnxruntime imageio-ffmpeg spconv-cu120 trimesh xatlas pyvista pymeshfix igraph",
  "pip install -q xformers --no-deps",
  "pip install -q git+https://github.com/NVlabs/nvdiffrast.git",
]
ok = True
for s in steps:
    left = 360 - (time.time() - t)
    if left <= 0:
        ok = False; done(result="skipped", reason="the install did not finish within 6 minutes"); break
    r = subprocess.run(s, shell=True, capture_output=True, text=True, timeout=left)
    print(s[:80], "->", r.returncode, r.stderr[-400:])
    if r.returncode != 0:
        ok = False; done(result="skipped", reason=f"install step failed: {s[:60]}", error=r.stderr[-800:]); break
status["install_seconds"] = round(time.time() - t)

In [ ]:
if ok:
    try:
        sys.path.insert(0, "/kaggle/working/TRELLIS")
        import torch
        from PIL import Image
        from trellis.pipelines import TrellisImageTo3DPipeline
        pipe = TrellisImageTo3DPipeline.from_pretrained("JeffreyXiang/TRELLIS-image-large"); pipe.cuda()
        made = []
        for name in ("temple_11", "ship_11"):
            src = glob.glob(f"/kaggle/input/**/concepts/{name}.png", recursive=True)[0]
            t1 = time.time()
            out = pipe.run(Image.open(src), seed=1)
            mesh = out["mesh"][0]
            import trimesh
            tm = trimesh.Trimesh(mesh.vertices.cpu().numpy(), mesh.faces.cpu().numpy())
            tm.export(f"/kaggle/working/trellis/{name}.obj")
            made.append({"name": name, "faces": len(tm.faces), "seconds": round(time.time() - t1, 1),
                         "peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 1)})
        done(result="ok", made=made)
    except Exception as err:
        done(result="failed", reason=f"{type(err).__name__}: {str(err)[:600]}")